# XGBoost 2.1+ Model — PharmShed Super Dataset

**Author:** Akhila Annireddy
**Model:** XGBoost 2.1+ multi-class classifier
**Task:** Multi-class classification — predict which of 218 classes a person is prescribed (217 drugs + "no prescriptions")
**Features:** Demographics (Age, Sex, Family_income, Insurance_coverage, Race_ethnicity) + Prescription (Quantity, Form, Strength, Day_Supply)
**Split strategy:** StratifiedGroupKFold (5-fold CV, shuffle=True, random_state=42), grouped by Person_ID to prevent data leakage
**Missing value strategy:**
  - Quantity NaNs: left as NaN — structurally identify 'no prescriptions' rows; XGBoost learns the missing direction natively.
  - Strength and Day_Supply NaNs: real missing values — imputed inside each fold using median grouped by year and drug.
  - Age NaNs: imputed inside each fold using hierarchical median (household → income/insurance group → year → global fallback).
  - Family_income NaNs: simple median imputation inside each fold on train split only.
  - Categorical features (Sex, Insurance_coverage, Race_ethnicity, Form): set to pandas category dtype; XGBoost handles missing categoricals natively.
**Metrics:** Accuracy, Cohen's Kappa, MCC, macro/micro averaged Precision, Recall, F1, F2
**Output:** Saves `xgboost_super_proba_2022.csv` — probability vector over 218 classes per observation, used as input to the ensemble model

## Key Design Decisions

**Why XGBoost 2.1+?**
XGBoost 2.1+ introduced native categorical support via `enable_categorical=True` on DMatrix,
eliminating the need for manual one-hot encoding. Gradient boosted trees are also naturally
robust to feature scale differences — no RobustScaler needed unlike KNN and SVM.

**Why leave Quantity NaNs as NaN?**
Unlike KNN and SVM which cannot handle NaN natively, XGBoost learns an optimal default
direction for missing values at each split. The 97,497 no-prescription rows have structurally
absent Quantity values — leaving them as NaN lets XGBoost learn this signal directly
rather than forcing it through a sentinel value.

**Why impute Strength and Day_Supply but not Quantity?**
Strength and Day_Supply have real missing values (not structural absences) — these are
genuinely unknown values where imputation adds signal. Quantity NaNs are structural
(all 97,497 no-prescription rows) so they carry discriminative meaning as NaN and should
not be imputed.

**Why hierarchical imputation for Age?**
Age missingness is not random — it correlates with household, income bracket, and insurance
coverage. A hierarchical strategy (household → income/insurance group → year → global)
produces more accurate imputations than a simple global median, reducing noise in the model.

**Why no class weights?**
XGBoost supports sample weights but full inverse frequency weighting collapsed after fold 1
in earlier experiments — the model over-corrected for rare classes and failed to converge.
Sqrt inverse frequency weighting was tested and improved macro recall by ~72% but threshold
tuning on the base model produced better results with less complexity. Class weighting is
therefore not applied in this version.

**Why `enable_categorical=True`?**
Setting categorical dtype on string columns and passing `enable_categorical=True` to DMatrix
tells XGBoost to use its native categorical split algorithm. This is more accurate than
ordinal encoding and avoids the dimensionality explosion of one-hot encoding.

In [ ]:
# Install required libraries.
# xgboost 2.1+ has native categorical support and handles missing values via
# learned default directions — no imputation needed for structural NaNs.
# permetrics provides our evaluation metrics.
!pip install 'xgboost>=2.1.0' permetrics

ERROR: Invalid requirement: "'xgboost": Expected package name at the start of dependency specifier
    'xgboost
    ^


In [ ]:
# Load all required libraries.
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

# Confirm versions for reproducibility
import sklearn
print('XGBoost version:     ', xgb.__version__)
print('pandas version:      ', pd.__version__)
print('numpy version:       ', np.__version__)
print('scikit-learn version:', sklearn.__version__)

# Hard stop if XGBoost version is below 2.1.0.
# Native categorical support and learned missing directions
# require XGBoost 2.1+ — earlier versions will silently produce wrong results.
from packaging import version
assert version.parse(xgb.__version__) >= version.parse('2.1.0'), \
    f"ERROR: XGBoost 2.1+ required, found {xgb.__version__} — run: pip install 'xgboost>=2.1.0'"

print('\nAll version checks passed.')

XGBoost version:      3.2.0
pandas version:       2.2.3
numpy version:        2.2.3
scikit-learn version: 1.6.1

All version checks passed.


In [ ]:
# Load the super integrated dataset (2014-2021).
# The super dataset contains demographics + prescription features
# (Quantity, Form, Strength, Day_Supply) + Person_ID.

DATA_DIR   = './'  # change this to your data path if needed
SUPER_DATA = f'{DATA_DIR}super_integrated_data.csv'

super_df = pd.read_csv(
    SUPER_DATA,
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)

# Drop auto-generated index column if present
if 'Unnamed: 0' in super_df.columns:
    super_df = super_df.drop(columns=['Unnamed: 0'])

print('Super dataset shape:', super_df.shape)
print('\nSuper dataset columns:', super_df.columns.tolist())
print('\nMissing values:')
print(super_df.isnull().sum())

Super dataset shape: (904140, 14)

Super dataset columns: ['Observation_ID', 'Person_ID', 'Household_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Year', 'Quantity', 'Form', 'Strength', 'Day_Supply']

Missing values:
Observation_ID             0
Person_ID                  0
Household_ID               0
Drug                       0
Age                     5261
Sex                        0
Family_income              0
Insurance_coverage         0
Race_ethnicity             0
Year                       0
Quantity               98066
Form                   98066
Strength               98146
Day_Supply            330224
dtype: int64


In [ ]:
# Verify Person_ID is present in the dataset.
# Person_ID is used only for StratifiedGroupKFold grouping — not a model feature.
assert 'Person_ID' in super_df.columns, \
    "ERROR: Person_ID not found in dataset — check super_integrated_data.csv."
assert super_df['Person_ID'].isnull().sum() == 0, \
    "ERROR: Missing Person_IDs — check super_integrated_data.csv."

print('Person_ID verified.')
print('Unique persons:', super_df['Person_ID'].nunique())

Person_ID verified.
Unique persons: 127415


In [ ]:
# EDA: confirm unique persons, drugs, distribution, missing values.
print('Unique persons:', super_df['Person_ID'].nunique())
print('Unique drugs:  ', super_df['Drug'].nunique())

print('\nTop 10 most prescribed drugs:')
print(super_df['Drug'].value_counts().head(10))

print('\nBottom 5 rarest drugs:')
print(super_df['Drug'].value_counts().tail(5))

print('\nMissing values per column:')
print(super_df.isnull().sum())

print('\nMissing value interpretation:')
print("  Quantity NaNs correspond to 'no prescriptions' rows — left as NaN for XGBoost native handling.")
print("  Strength / Day_Supply NaNs: real missing values — imputed inside each fold.")
print("  'no prescriptions' count:", (super_df['Drug'] == 'no prescriptions').sum())

# Check class imbalance ratio — max count / min count.
# Confirms severe imbalance and justifies reporting macro metrics
# which treat rare and common drugs equally — not just overall accuracy.
counts = super_df['Drug'].value_counts()
print(f'\nMost common drug count:  {counts.max():,}')
print(f'Rarest drug count:       {counts.min():,}')
print(f'Imbalance ratio:         {counts.max() / counts.min():.1f}x')

# Hard stop if drug count is not 217 (216 drugs + "no prescriptions").
assert super_df['Drug'].nunique() == 217, \
    f"ERROR: Expected 217 drug classes, found {super_df['Drug'].nunique()}."

print('\nDrug class count confirmed: 217 (216 drugs + no prescriptions).')

Unique persons: 127415
Unique drugs:   217

Top 10 most prescribed drugs:
Drug
no prescriptions    98066
atorvastatin        38542
lisinopril          35831
metformin           33774
amlodipine          28130
metoprolol          24981
albuterol           23188
omeprazole          22730
losartan            18670
gabapentin          18335
Name: count, dtype: int64

Bottom 5 rarest drugs:
Drug
gentamicin          64
piroxicam           61
sulfamethoxazole    57
trimethoprim        57
ivermectin          29
Name: count, dtype: int64

Missing values per column:
Observation_ID             0
Person_ID                  0
Household_ID               0
Drug                       0
Age                     5261
Sex                        0
Family_income              0
Insurance_coverage         0
Race_ethnicity             0
Year                       0
Quantity               98066
Form                   98066
Strength               98146
Day_Supply            330224
dtype: int64

Missing value int

## Preprocessing for XGBoost 2.1+ — Super Dataset

**Three-tier missing value strategy:**

1. **Quantity NaNs** — left as `NaN`.
   Quantity NaNs correspond to 'no prescriptions' rows and are structurally absent — not unknown.
   XGBoost's sparsity-aware split-finding algorithm learns an optimal default branch direction
   for missing values at every tree node, routing these rows to the correct leaf automatically.
   Replacing these NaNs with any value (median, -1, 0) would destroy this structural signal.

2. **Strength and Day_Supply NaNs** — imputed **inside the CV loop** using median imputation
   grouped by year and drug. These are real missing values (not structural absences) so they
   require explicit imputation. Imputation is applied on train and val separately to prevent
   data leakage.

3. **Age NaNs** — imputed **inside the CV loop** using hierarchical median imputation
   (household → income/insurance group → year → global fallback). Fit on train split only,
   applied to val split to prevent leakage.

4. **Family_income NaNs** — simple median imputation inside the CV loop on train split only.

5. **Categorical columns (Sex, Insurance_coverage, Race_ethnicity, Form)** — set to `category` dtype.
   XGBoost 2.1+ handles missing categoricals natively via `enable_categorical=True` in DMatrix.

**XGBoost does not require feature scaling** — numeric columns are passed as-is.
Unlike KNN (distance-based) and SVM (margin-based), XGBoost tree splits are scale-invariant.
RobustScaler is intentionally not applied.

**Key difference from KNN and SVM:**
Quantity NaNs are left as NaN here — not filled with -1 sentinel.
KNN and SVM cannot handle NaN natively so they required explicit filling.
XGBoost uses Quantity NaN as a meaningful structural signal and learns from it directly.
Strength and Day_Supply are imputed here (unlike KNN/SVM) because XGBoost can handle
structural NaNs but benefits from imputed real missing values for better split quality.

In [ ]:
# Define columns and fit LabelEncoder once on the full dataset.
# LabelEncoder is fitted before the CV loop so the drug->integer mapping
# is identical across all folds and for the final model.
feature_cols          = ['Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity',
                         'Quantity', 'Form', 'Strength', 'Day_Supply']
categorical_cols      = ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
numeric_demo_cols     = ['Age', 'Family_income']             # imputed inside CV loop
numeric_rx_cols       = ['Quantity', 'Strength', 'Day_Supply']  # Quantity left as NaN; Strength/Day_Supply imputed inside CV loop
target_col            = 'Drug'

le = LabelEncoder()
super_df['Drug_encoded'] = le.fit_transform(super_df[target_col])

print('Unique classes in encoder:', len(le.classes_))
print('Feature columns:          ', feature_cols)
print('Categorical cols:         ', categorical_cols)
print('Numeric demo cols:        ', numeric_demo_cols)
print('Numeric Rx cols:          ', numeric_rx_cols)
print('\nSample drug->integer mapping (first 5):')
for i, drug in enumerate(le.classes_[:5]):
    print(f'  {drug} -> {i}')

# Hard stop if any defined column is missing from the dataset.
# Catches typos in column names or dataset changes before the CV loop starts.
missing_cols = [c for c in feature_cols + [target_col] if c not in super_df.columns]
assert len(missing_cols) == 0, \
    f"ERROR: These columns are missing from the dataset: {missing_cols}"

# Save the LabelEncoder so it can be reloaded without rerunning this notebook.
# Required for ensemble model — all models must use identical drug-to-integer mappings.
joblib.dump(le, 'xgboost_super_label_encoder.joblib')
print('\nLabelEncoder saved to xgboost_super_label_encoder.joblib')

Unique classes in encoder: 217
Feature columns:           ['Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
Categorical cols:          ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
Numeric demo cols:         ['Age', 'Family_income']
Numeric Rx cols:           ['Quantity', 'Strength', 'Day_Supply']

Sample drug->integer mapping (first 5):
  acetaminophen -> 0
  acyclovir -> 1
  adapalene -> 2
  albuterol -> 3
  alendronate -> 4

LabelEncoder saved to xgboost_super_label_encoder.joblib


In [ ]:
# set structural NaN's to -1 (this is necessary to differentiate between random
# missingness and the structural missingness of "no prescriptions")
mask = super_df['Drug'] == 'no prescriptions'
super_df.loc[mask, numeric_rx_cols] = -1
super_df.loc[mask, 'Form'] = '-1'

In [ ]:
# Imputation functions for Age, Strength, and Day_Supply.
# Applied within each fold to prevent data leakage.
# Consistent with TabICL and all other base models.
# Note: Quantity NaNs are NOT imputed — they are structurally meaningful
# and left as NaN for XGBoost native handling.

def impute_age(df):
    df = df.copy()
    df['income_bracket'] = (
        df.groupby('Year')['Family_income']
        .transform(lambda x: pd.qcut(x, 4, labels=False, duplicates='drop') + 1)
    )
    hh_meds    = df.groupby(['Year', 'Household_ID'])['Age'].transform('median')
    grp_meds   = df.groupby(['Year', 'income_bracket', 'Insurance_coverage'])['Age'].transform('median')
    yr_meds    = df.groupby('Year')['Age'].transform('median')
    global_med = df['Age'].median()
    df['Age']  = df['Age'].fillna(hh_meds)
    df['Age']  = df['Age'].fillna(grp_meds)
    df['Age']  = df['Age'].fillna(yr_meds)
    df['Age']  = df['Age'].fillna(global_med)
    df.drop(columns=['income_bracket'], inplace=True)
    return df

def impute_strength(df):
    df = df.copy()
    mask = df['Drug'] != 'no prescriptions'
    # 1. Median by Year and Drug
    yr_drug_meds = df.groupby(['Year', 'Drug'])['Strength'].transform('median')
    # 2. Median by Drug (across all years)
    drug_meds = df.groupby('Drug')['Strength'].transform('median')
    # 3. Global Median
    global_med   = df.loc[mask, 'Strength'].median()

    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(yr_drug_meds)
    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(drug_meds)
    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(global_med)
    return df

def impute_day_supply(df):
    df = df.copy()
    mask = df['Drug'] != 'no prescriptions'
    # 1. Median by Year and Drug
    yr_drug_meds = df.groupby(['Year', 'Drug'])['Day_Supply'].transform('median')
    # 2. Median by Drug
    drug_meds = df.groupby('Drug')['Day_Supply'].transform('median')
    # 3. Global Median
    global_med   = df.loc[mask, 'Day_Supply'].median()

    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(yr_drug_meds)
    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(drug_meds)
    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(global_med)
    return df

In [ ]:
# StratifiedGroupKFold 5-fold CV for XGBoost Super Dataset.
#
# For each fold:
#   1. Split by Person_ID groups, stratified by Drug
#   2. Impute Age, Strength, Day_Supply within each fold (fit on train, apply to val)
#   3. Simple median imputation for Family_income on train split only
#   4. Set categorical dtypes for XGBoost native handling
#   5. Convert to DMatrix with enable_categorical=True
#      (Quantity NaNs remain as NaN — XGBoost learns default direction)
#   6. Train XGBoost multi:softmax classifier with early stopping
#   7. Predict and compute all required metrics

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

X      = super_df[feature_cols].copy()
y      = super_df['Drug_encoded'].values
groups = super_df['Person_ID'].values

# XGBoost parameters.
# multi:softmax outputs predicted class integers directly.
# tree_method='hist' is fast and memory-efficient for large datasets.
# device: automatically set — 'cuda' if NVIDIA GPU available, else 'cpu'.
# N_ROUNDS=100: early stopping will find the true optimal round per fold.
# early_stopping_rounds=20: stops if val error does not improve for 20 rounds.
XGB_DEVICE = 'cuda' if xgb.build_info().get('USE_CUDA', False) else 'cpu'
print(f'XGBoost device: {XGB_DEVICE}')

xgb_params = {
    'objective':        'multi:softprob',
    'num_class':        len(le.classes_),
    'eval_metric':      'mlogloss',
    'device':           XGB_DEVICE,
    'tree_method':      'hist',
    'max_depth':        6,
    'learning_rate':    0.1,
    'subsample':        0.8,
    'colsample_bytree': 1.0,
    'random_state':     42,
    'verbosity':        1,
}
N_ROUNDS       = 100
EARLY_STOPPING = 20

fold_results          = []
per_drug_recall_folds = []

for fold_num, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    print(f'\n{"="*50}')
    print(f'FOLD {fold_num}/5')
    print(f'{"="*50}')

    # Split on super_df so imputation functions have access to all columns
    train_fold = super_df.iloc[train_idx].copy()
    val_fold   = super_df.iloc[val_idx].copy()

    # Impute Age using hierarchical median strategy (household → income/insurance → year → global).
    # Impute Strength and Day_Supply using median grouped by year and drug.
    # All imputation fit on train and applied to val to prevent data leakage.
    train_fold = impute_age(train_fold)
    val_fold   = impute_age(val_fold)

    train_fold = impute_strength(train_fold)
    val_fold   = impute_strength(val_fold)

    train_fold = impute_day_supply(train_fold)
    val_fold   = impute_day_supply(val_fold)

    X_train_fold = train_fold[feature_cols].copy()
    X_val_fold   = val_fold[feature_cols].copy()
    y_train_fold = train_fold['Drug_encoded'].values
    y_val_fold   = val_fold['Drug_encoded'].values

    # Set categorical dtypes — XGBoost handles encoding and missing values internally.
    for col in categorical_cols:
        X_train_fold[col] = X_train_fold[col].astype('category')
        X_val_fold[col]   = X_val_fold[col].astype('category')

    print(f'Train size: {len(X_train_fold):,} | Val size: {len(X_val_fold):,}')
    print(f'Unique drugs — train: {len(np.unique(y_train_fold))} | val: {len(np.unique(y_val_fold))}')

    # Convert to DMatrix with enable_categorical=True.
    # Quantity NaNs are passed through as NaN — XGBoost learns the optimal
    # split direction for missing values at each node.
    dtrain = xgb.DMatrix(X_train_fold, label=y_train_fold, enable_categorical=True)
    dval   = xgb.DMatrix(X_val_fold,   label=y_val_fold,   enable_categorical=True)

    # Train
    evals_result = {}
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=N_ROUNDS,
        evals=[(dtrain, 'train'), (dval, 'val')],
        evals_result=evals_result,
        verbose_eval=10,
        early_stopping_rounds=EARLY_STOPPING,
    )
    print(f'Fold {fold_num} training complete. Best round: {model.best_iteration}')

    # Predict
    #y_pred_fold = model.predict(dval).astype(int)
    # get it out of integer mode (expected indices from softmax but now softprob)
    proba_fold = model.predict(dval)
    # Since dval is a DMatrix, this returns the (N, 217) matrix automatically.
    # To get the metrics (Accuracy, etc.), we take the argmax:
    y_pred_fold = np.argmax(proba_fold, axis=1)

    # Metrics
    acc   = accuracy_score(y_val_fold, y_pred_fold)
    kappa = cohen_kappa_score(y_val_fold, y_pred_fold)
    mcc   = matthews_corrcoef(y_val_fold, y_pred_fold)

    evaluator       = ClassificationMetric(y_val_fold, y_pred_fold)
    macro_precision = evaluator.precision_score(average='macro')
    micro_precision = evaluator.precision_score(average='micro')
    macro_recall    = evaluator.recall_score(average='macro')
    micro_recall    = evaluator.recall_score(average='micro')
    macro_f1        = evaluator.f1_score(average='macro')
    micro_f1        = evaluator.f1_score(average='micro')
    macro_f2        = evaluator.fbeta_score(beta=2, average='macro')
    micro_f2        = evaluator.fbeta_score(beta=2, average='micro')

    fold_results.append({
        'fold':             fold_num,
        'accuracy':         acc,
        'cohen_kappa':      kappa,
        'mcc':              mcc,
        'macro_precision':  macro_precision,
        'micro_precision':  micro_precision,
        'macro_recall':     macro_recall,
        'micro_recall':     micro_recall,
        'macro_f1':         macro_f1,
        'micro_f1':         micro_f1,
        'macro_f2':         macro_f2,
        'micro_f2':         micro_f2,
    })

    # Per-drug recall for ensemble model selection
    report = classification_report(
        y_val_fold, y_pred_fold,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_recalls         = {drug: report[drug]['recall'] for drug in le.classes_ if drug in report}
    drug_recalls['fold'] = fold_num
    per_drug_recall_folds.append(drug_recalls)

    print(f'Fold {fold_num} results:')
    print(f'  Accuracy:     {acc:.4f}')
    print(f'  Cohen Kappa:  {kappa:.4f}')
    print(f'  MCC:          {mcc:.4f}')
    print(f'  Macro Recall: {macro_recall:.4f}')
    print(f'  Micro Recall: {micro_recall:.4f}')
    print(f'  Macro F2:     {macro_f2:.4f}')

print('\n' + '='*50)
print('ALL FOLDS COMPLETE')
print('='*50)

XGBoost device: cuda

FOLD 1/5
Before age: 15
After age: 15
Train size: 726,282 | Val size: 177,858
Unique drugs — train: 217 | val: 217
[0]	train-merror:0.64993	val-merror:0.65756
[10]	train-merror:0.44693	val-merror:0.47825
[20]	train-merror:0.42627	val-merror:0.46700
[30]	train-merror:0.41460	val-merror:0.46257
[40]	train-merror:0.40469	val-merror:0.46045
[50]	train-merror:0.39586	val-merror:0.45936
[60]	train-merror:0.38827	val-merror:0.45801
[70]	train-merror:0.38090	val-merror:0.45607
[80]	train-merror:0.37358	val-merror:0.45659
[90]	train-merror:0.36564	val-merror:0.45599
[99]	train-merror:0.35858	val-merror:0.45575
Fold 1 training complete. Best round: 89
Fold 1 results:
  Accuracy:     0.5443
  Cohen Kappa:  0.5308
  MCC:          0.5325
  Macro Recall: 0.4536
  Micro Recall: 0.5443
  Macro F2:     0.4536

FOLD 2/5
Before age: 15
After age: 15
Train size: 723,019 | Val size: 181,121
Unique drugs — train: 217 | val: 217
[0]	train-merror:0.65054	val-merror:0.65257
[10]	train-mer

In [ ]:
# Summarize CV results — mean and std per metric across all 5 folds.
# These numbers go directly into the paper.

results_df = pd.DataFrame(fold_results)

# Hard stop if not all 5 folds completed.
# If the CV loop crashed mid-run, this catches it before saving incomplete results.
assert len(results_df) == 5, \
    f"ERROR: Expected 5 fold results, found {len(results_df)} — CV may not have completed."

print('Per-fold results:')
print(results_df.to_string(index=False))

print('\nMean ± Std across 5 folds:')
metric_cols = [c for c in results_df.columns if c != 'fold']
for col in metric_cols:
    mean = results_df[col].mean()
    std  = results_df[col].std()
    print(f'  {col:25s}: {mean:.4f} ± {std:.4f}')

results_df.to_csv('xgboost_super_cv_results.csv', index=False)
print('\nCV results saved to xgboost_super_cv_results.csv')
print('All 5 folds confirmed complete.')

Per-fold results:
 fold  accuracy  cohen_kappa      mcc  macro_precision  micro_precision  macro_recall  micro_recall  macro_f1  micro_f1  macro_f2  micro_f2
    1  0.544254     0.530762 0.532526         0.502758         0.544254      0.453573      0.544254  0.459698  0.544254  0.453644  0.544254
    2  0.547231     0.534380 0.536207         0.509366         0.547231      0.458230      0.547231  0.462398  0.547231  0.457140  0.547231
    3  0.541524     0.528140 0.529897         0.508012         0.541524      0.456222      0.541524  0.462657  0.541524  0.456292  0.541524
    4  0.545938     0.532950 0.534523         0.518447         0.545938      0.460132      0.545938  0.470161  0.545938  0.461599  0.545938
    5  0.541765     0.528495 0.530199         0.505832         0.541765      0.450956      0.541765  0.459928  0.541765  0.452272  0.541765

Mean ± Std across 5 folds:
  accuracy                 : 0.5441 ± 0.0025
  cohen_kappa              : 0.5309 ± 0.0027
  mcc                   

In [ ]:
# Average per-drug recall across all 5 folds.
# This CSV goes for ensemble construction.

per_drug_df = pd.DataFrame(per_drug_recall_folds)
drug_cols   = [c for c in per_drug_df.columns if c != 'fold']

mean_drug_recall = per_drug_df[drug_cols].mean().reset_index()
mean_drug_recall.columns = ['Drug', 'Mean_Recall_XGBoost_Super']
mean_drug_recall = mean_drug_recall.sort_values('Mean_Recall_XGBoost_Super', ascending=False)

print('Top 20 drugs by mean recall (super dataset):')
print(mean_drug_recall.head(20).to_string(index=False))

print('\nBottom 20 drugs by mean recall (super dataset):')
print(mean_drug_recall.tail(20).to_string(index=False))

# Summary stats — how many drugs does XGBoost Super recall?
# Matches the format used across all other model notebooks for direct comparison.
drugs_recalled = (mean_drug_recall['Mean_Recall_XGBoost_Super'] > 0).sum()
print(f'\nDrugs with mean recall > 0:    {drugs_recalled} / {len(mean_drug_recall)}')
print(f'Drugs with mean recall >= 0.1: {(mean_drug_recall["Mean_Recall_XGBoost_Super"] >= 0.1).sum()}')
print(f'Drugs with mean recall >= 0.5: {(mean_drug_recall["Mean_Recall_XGBoost_Super"] >= 0.5).sum()}')

mean_drug_recall.to_csv('xgboost_super_per_drug_recall.csv', index=False)
print('\nPer-drug recall saved to xgboost_super_per_drug_recall.csv')

Top 20 drugs by mean recall (super dataset):
            Drug  Mean_Recall_XGBoost_Super
      colchicine                   1.000000
no prescriptions                   1.000000
       lactulose                   1.000000
      tiotropium                   0.998464
     latanoprost                   0.998299
      tamsulosin                   0.997384
     liraglutide                   0.995425
     linaclotide                   0.995152
       albuterol                   0.995146
     fluticasone                   0.989868
       clonidine                   0.989163
     dorzolamide                   0.988624
     bimatoprost                   0.981000
      azelastine                   0.980138
     alendronate                   0.976007
         aspirin                   0.973436
    moxifloxacin                   0.972324
       metformin                   0.968949
   nitroglycerin                   0.964170
   chlorhexidine                   0.955412

Bottom 20 drugs by mean recall

## Final Model — Train on Full Dataset (2014–2021)

After CV confirms performance, train one final model on the full dataset for internal
validation on MEPS 2022 and ensemble use.

For the final model:
- Age is imputed using the hierarchical median imputation function on the full dataset
- Strength and Day_Supply are imputed using median grouped by year and drug on the full dataset
- Family_income is median-imputed on the full training set
- Quantity NaNs remain as NaN — XGBoost learns the missing direction natively

In [ ]:
# Train final XGBoost model on the full super integrated dataset (2014-2021).

# Impute Age, Strength, Day_Supply on full dataset before final training.
data_final = impute_age(super_df.copy())
data_final = impute_strength(data_final)
data_final = impute_day_supply(data_final)

X_final = data_final[feature_cols].copy()
y_final = data_final['Drug_encoded'].values

for col in categorical_cols:
    X_final[col] = X_final[col].astype('category')

# Shuffle to separate refill records after imputation.
X_final, y_final = shuffle(X_final, y_final, random_state=42)
X_final = X_final.reset_index(drop=True)

print('Final training data size:', X_final.shape)
print('Number of classes:       ', len(le.classes_))
print(f'Device:                   {XGB_DEVICE}')

# Quantity NaNs passed through as NaN — XGBoost learns missing direction natively.
dtrain_final = xgb.DMatrix(X_final, label=y_final, enable_categorical=True)

print('\nTraining final XGBoost model on super dataset...')
final_model = xgb.train(
    xgb_params,
    dtrain_final,
    num_boost_round=N_ROUNDS,
    verbose_eval=10,
)
print('Final model training complete.')

# Save final model in XGBoost binary format.
# .ubj is XGBoost's recommended binary format — smaller and faster than JSON.
# Can be reloaded with: model = xgb.Booster(); model.load_model('xgboost_super_final_model.ubj')
final_model.save_model('xgboost_super_final_model.ubj')
print('Model saved to xgboost_super_final_model.ubj')

# Save Family_income median — must be saved alongside the model.
# 2022 validation data must use this exact median for imputation.
import json
#with open('xgboost_super_train_medians.json', 'w') as f:
 #   json.dump({'Family_income': final_family_income_median}, f)
#print('Train medians saved to xgboost_super_train_medians.json')

Final model Family_income median: 38940.0
Final training data size: (904140, 9)
Number of classes:        217
Device:                   cuda

Training final XGBoost model on super dataset...
Final model training complete.
Model saved to xgboost_super_final_model.ubj
Train medians saved to xgboost_super_train_medians.json


In [ ]:
# Internal validation on MEPS 2022 (held-out test set).
# 2022 is never seen during training or CV.
#
# Preprocessing must mirror training exactly:
#   - Age, Strength, Day_Supply: imputed using same functions as training
#   - Family_income: filled with final_family_income_median (computed on 2014-2021)
#   - Quantity NaNs: left as NaN (same as training — XGBoost handles natively)
#   - Categorical cols: set to category dtype

data_2022 = pd.read_csv(
    f'{DATA_DIR}super_data_2022.csv',
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)
if 'Unnamed: 0' in data_2022.columns:
    data_2022 = data_2022.drop(columns=['Unnamed: 0'])

print('2022 super data shape:', data_2022.shape)

# Filter to only drugs the model knows
known_drugs  = set(le.classes_)
unseen_drugs = set(data_2022['Drug'].unique()) - known_drugs
print(f'Unseen drugs in 2022 (will be dropped): {len(unseen_drugs)}')
if unseen_drugs:
    print('Unseen:', unseen_drugs)

data_2022_filtered = data_2022[data_2022['Drug'].isin(known_drugs)].copy()
print(f'2022 rows after filtering: {len(data_2022_filtered):,}')

# fill sentinel value for structural NaNs
mask = data_2022_filtered['Drug'] == 'no prescriptions'
data_2022_filtered.loc[mask, numeric_rx_cols] = data_2022_filtered.loc[mask, numeric_rx_cols].fillna(-1)
data_2022_filtered.loc[mask, 'Form'] = data_2022_filtered.loc[mask, 'Form'].fillna('-1')

# Impute Age, Strength, Day_Supply — must mirror training preprocessing.
data_2022_filtered = impute_age(data_2022_filtered)
data_2022_filtered = impute_strength(data_2022_filtered)
data_2022_filtered = impute_day_supply(data_2022_filtered)

# Apply training Family_income median — do NOT recompute from 2022 data.
# Using 2022 median would leak test statistics into preprocessing.
#data_2022_filtered['Family_income'] = data_2022_filtered['Family_income'].fillna(final_family_income_median)

X_2022 = data_2022[feature_cols].copy()
for col in categorical_cols:
    X_2022[col] = X_2022[col].astype('category')

y_2022_encoded = le.transform(data_2022_filtered['Drug'])

# Shuffle to separate refill records.
X_2022, y_2022_encoded = shuffle(X_2022, y_2022_encoded, random_state=42)
X_2022 = X_2022.reset_index(drop=True)

# Quantity NaNs passed through as NaN — XGBoost learns missing direction natively.
d2022 = xgb.DMatrix(X_2022, label=y_2022_encoded, enable_categorical=True)

#y_pred_2022 = final_model.predict(d2022).astype(int)
# get it out of integer mode (expected indices from softmax but now softprob)
proba_2022_raw = final_model.predict(d2022)
y_pred_2022 = np.argmax(proba_2022_raw, axis=1)

# Metrics
acc_2022   = accuracy_score(y_2022_encoded, y_pred_2022)
kappa_2022 = cohen_kappa_score(y_2022_encoded, y_pred_2022)
mcc_2022   = matthews_corrcoef(y_2022_encoded, y_pred_2022)

ev2022            = ClassificationMetric(y_2022_encoded, y_pred_2022)
macro_prec_2022   = ev2022.precision_score(average='macro')
micro_prec_2022   = ev2022.precision_score(average='micro')
macro_recall_2022 = ev2022.recall_score(average='macro')
micro_recall_2022 = ev2022.recall_score(average='micro')
macro_f2_2022     = ev2022.fbeta_score(beta=2, average='macro')
micro_f2_2022     = ev2022.fbeta_score(beta=2, average='micro')

print('\nINTERNAL VALIDATION — MEPS 2022 Results (Super Dataset)')
print(f'Accuracy:          {acc_2022:.4f}')
print(f'Cohen Kappa:       {kappa_2022:.4f}')
print(f'MCC:               {mcc_2022:.4f}')
print(f'Macro Precision:   {macro_prec_2022:.4f}')
print(f'Micro Precision:   {micro_prec_2022:.4f}')
print(f'Macro Recall:      {macro_recall_2022:.4f}')
print(f'Micro Recall:      {micro_recall_2022:.4f}')
print(f'Macro F2:          {macro_f2_2022:.4f}')
print(f'Micro F2:          {micro_f2_2022:.4f}')

# Per-drug metrics on 2022
report_2022 = classification_report(
    y_2022_encoded, y_pred_2022,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)
drug_metrics_2022 = pd.DataFrame([
    {'Drug':           drug,
     'Recall_2022':    report_2022[drug]['recall'],
     'Precision_2022': report_2022[drug]['precision'],
     'F1_2022':        report_2022[drug]['f1-score'],
     'Support_2022':   report_2022[drug]['support']}
    for drug in le.classes_ if drug in report_2022
]).sort_values('Recall_2022', ascending=False)

print('\nTop 15 drugs by recall on 2022:')
print(drug_metrics_2022.head(15).to_string(index=False))
print('\nBottom 15 drugs by recall on 2022:')
print(drug_metrics_2022.tail(15).to_string(index=False))

# Summary stats
drugs_recalled_2022 = (drug_metrics_2022['Recall_2022'] > 0).sum()
print(f'\nDrugs recalled on 2022 data (recall > 0): {drugs_recalled_2022} / {len(drug_metrics_2022)}')
print(f'Drugs with recall >= 0.1: {(drug_metrics_2022["Recall_2022"] >= 0.1).sum()}')
print(f'Drugs with recall >= 0.5: {(drug_metrics_2022["Recall_2022"] >= 0.5).sum()}')

drug_metrics_2022.to_csv('xgboost_super_2022_per_drug_metrics.csv', index=False)
print('\nPer-drug 2022 metrics saved to xgboost_super_2022_per_drug_metrics.csv')

pd.DataFrame([{
    'model':            'XGBoost_Super',
    'dataset':          'MEPS_2022_internal_validation',
    'accuracy':         acc_2022,
    'cohen_kappa':      kappa_2022,
    'mcc':              mcc_2022,
    'macro_precision':  macro_prec_2022,
    'micro_precision':  micro_prec_2022,
    'macro_recall':     macro_recall_2022,
    'micro_recall':     micro_recall_2022,
    'macro_f2':         macro_f2_2022,
    'micro_f2':         micro_f2_2022,
}]).to_csv('xgboost_super_validation_summary.csv', index=False)
print('Validation summary saved to xgboost_super_validation_summary.csv')

# Save probability outputs for ensemble model.
# XGBoost with multi:softmax outputs hard labels from predict().
# To get probabilities we switch objective to multi:softprob temporarily.
# Shape: (n_observations, 218) — one row per observation, one column per drug class.
#xgb_params_proba = xgb_params.copy()
#xgb_params_proba['objective'] = 'multi:softprob'

#print('\nGenerating probability outputs for ensemble...')
#final_model_proba = xgb.train(
 #   xgb_params_proba,
  #  xgb.DMatrix(X_final, label=y_final, enable_categorical=True),
   # num_boost_round=N_ROUNDS,
    #verbose_eval=False,
#)
print('\nSaving probability outputs for ensemble...')
# proba_2022_raw was already created in the validation step above
proba_df = pd.DataFrame(proba_2022_raw, columns=le.classes_)
proba_df.insert(0, 'Observation_ID', data_2022_filtered['Observation_ID'].reset_index(drop=True).values)
proba_df.to_csv('xgboost_super_proba_2022.csv', index=False)
print(f'Probability output saved to xgboost_super_proba_2022.csv')
print(f'Shape: {proba_df.shape} — {len(X_2022):,} observations x 218 classes + Observation_ID')

2022 super data shape: (175477, 14)
Unseen drugs in 2022 (will be dropped): 0
2022 rows after filtering: 175,477

INTERNAL VALIDATION — MEPS 2022 Results (Super Dataset)
Accuracy:          0.5317
Cohen Kappa:       0.5202
MCC:               0.5225
Macro Precision:   0.4962
Micro Precision:   0.5317
Macro Recall:      0.4459
Micro Recall:      0.5317
Macro F2:          0.4434
Micro F2:          0.5317

Top 15 drugs by recall on 2022:
            Drug  Recall_2022  Precision_2022  F1_2022  Support_2022
     bimatoprost     1.000000        1.000000 1.000000         156.0
     clavulanate     1.000000        1.000000 1.000000          16.0
      colchicine     1.000000        1.000000 1.000000         168.0
    cyclosporine     1.000000        0.990909 0.995434         218.0
     latanoprost     1.000000        0.975764 0.987733         926.0
no prescriptions     1.000000        1.000000 1.000000       10129.0
       lactulose     1.000000        1.000000 1.000000          90.0
     liragl

In [ ]:
# Inspect prediction distribution on 2022 data.
# ratio > 1 = over-predicted, ratio < 1 = under-predicted, 0 = never predicted.
# Compare with base XGBoost to assess whether prescription features improve rare drug recall.

pred_drugs_2022   = le.inverse_transform(y_pred_2022)
actual_drugs_2022 = le.inverse_transform(y_2022_encoded)

pred_counts   = pd.Series(pred_drugs_2022).value_counts().rename('predicted')
actual_counts = pd.Series(actual_drugs_2022).value_counts().rename('actual')

dist_compare = pd.concat([actual_counts, pred_counts], axis=1).fillna(0).astype(int)
dist_compare['ratio_pred_to_actual'] = (
    dist_compare['predicted'] / dist_compare['actual'].replace(0, 1)
).round(2)
dist_compare = dist_compare.sort_values('actual', ascending=False)

print('Prediction vs actual (top 20 most common drugs):')
print(dist_compare.head(20))
print('\nPrediction vs actual (bottom 20 rarest drugs):')
print(dist_compare.tail(20))

never_predicted = dist_compare[dist_compare['predicted'] == 0]
print(f'\nDrugs never predicted: {len(never_predicted)}')
if len(never_predicted) > 0:
    print(never_predicted.index.tolist())

# Over-prediction check — drugs predicted far more than actual.
# High ratio means the model is over-relying on common drugs.
# Ratio > 5 is a signal worth investigating.
over_predicted = dist_compare[dist_compare['ratio_pred_to_actual'] > 5]
print(f'\nDrugs over-predicted by 5x or more: {len(over_predicted)}')
if len(over_predicted) > 0:
    print(over_predicted[['actual', 'predicted', 'ratio_pred_to_actual']])

Prediction vs actual (top 20 most common drugs):
                     actual  predicted  ratio_pred_to_actual
no prescriptions      10129      10129                  1.00
atorvastatin           9564      21383                  2.24
metformin              7039       8275                  1.18
lisinopril             6735       5045                  0.75
amlodipine             6245      13763                  2.20
metoprolol             5426       6856                  1.26
albuterol              5057       5055                  1.00
losartan               4307       5849                  1.36
omeprazole             4259       6617                  1.55
gabapentin             3832       4329                  1.13
hydrochlorothiazide    3327       6915                  2.08
rosuvastatin           2985         44                  0.01
sertraline             2827       3266                  1.16
pantoprazole           2543       1828                  0.72
montelukast            2288       40

In [ ]:
# Feature importance from XGBoost (by gain).
# With prescription features now included, we expect Quantity, Strength, Day_Supply
# to rank highly — they are drug-specific and directly discriminative.
# Compare the ranking shift vs base XGBoost (demographics only) for the paper discussion.

importance = final_model.get_score(importance_type='gain')
importance_df = pd.DataFrame([
    {'Feature': k, 'Importance_Gain': v}
    for k, v in importance.items()
]).sort_values('Importance_Gain', ascending=False)

print('Feature importance by gain (super dataset — demographics + prescription):')
print(importance_df.to_string(index=False))

importance_df.to_csv('xgboost_super_feature_importance.csv', index=False)
print('\nFeature importance saved to xgboost_super_feature_importance.csv')

# Final output file summary — confirms everything saved correctly.
print('\n' + '='*55)
print('ALL OUTPUTS SAVED')
print('='*55)
print('  xgboost_super_cv_results.csv')
print('  xgboost_super_per_drug_recall.csv')
print('  xgboost_super_2022_per_drug_metrics.csv')
print('  xgboost_super_validation_summary.csv')
print('  xgboost_super_proba_2022.csv             <- ensemble input')
print('  xgboost_super_final_model.ubj')
print('  xgboost_super_train_medians.json')
print('  xgboost_super_label_encoder.joblib')
print('  xgboost_super_feature_importance.csv')

Feature importance by gain (super dataset — demographics + prescription):
           Feature  Importance_Gain
              Form      1300.182007
          Strength       339.823822
        Day_Supply       207.378769
          Quantity       200.443481
               Sex        95.585175
               Age        26.956072
Insurance_coverage        22.917877
     Family_income        14.155286
    Race_ethnicity        12.783838

Feature importance saved to xgboost_super_feature_importance.csv

ALL OUTPUTS SAVED
  xgboost_super_cv_results.csv
  xgboost_super_per_drug_recall.csv
  xgboost_super_2022_per_drug_metrics.csv
  xgboost_super_validation_summary.csv
  xgboost_super_proba_2022.csv             <- ensemble input
  xgboost_super_final_model.ubj
  xgboost_super_train_medians.json
  xgboost_super_label_encoder.joblib
  xgboost_super_feature_importance.csv


## Summary of Outputs

| File | Contents |
|------|----------|
| `xgboost_super_cv_results.csv` | Mean ± std for all metrics across 5 CV folds |
| `xgboost_super_per_drug_recall.csv` | Average recall per drug across 5 folds (for ensemble) |
| `xgboost_super_2022_per_drug_metrics.csv` | Per-drug recall, precision, F1 on MEPS 2022 |
| `xgboost_super_validation_summary.csv` | Overall validation metrics on MEPS 2022 |
| `xgboost_super_proba_2022.csv` | Probability vector over 218 classes per observation — input to ensemble model |
| `xgboost_super_feature_importance.csv` | Feature importance by gain (demographics + prescription features) |
| `xgboost_super_final_model.ubj` | Saved final XGBoost model |
| `xgboost_super_train_medians.json` | Family_income median from training data for 2022 validation preprocessing |
| `xgboost_super_label_encoder.joblib` | Saved LabelEncoder for drug class mapping |

**Key difference from base XGBoost:** Prescription features (Quantity, Form, Strength, Day_Supply) are added. Quantity NaNs are left intact for XGBoost native handling — they structurally identify 'no prescriptions' rows. Strength and Day_Supply are imputed using median grouped by year and drug. Age uses hierarchical median imputation. Family_income uses simple median imputation.

**Next step:** Pass `xgboost_super_proba_2022.csv` to the ensemble model (`ensemble_pharmshed.ipynb`) along with probability outputs from all other base models for soft voting ensemble construction.